In [ ]:

try:
    import kaggle_benchmarks as kbench
except ImportError:
    # Failsafe: Mock the kbench harness if the library is not found (e.g. during build/commit)
    import types
    class MockTask:
        def __init__(self, f): self.f = f
        def run(self, *args, **kwargs): return self.f(*args, **kwargs)
        def evaluate(self, *args, **kwargs):
            class Res: 
                def as_dataframe(self): return None
            return Res()
        def __call__(self, *args, **kwargs): return self.f(*args, **kwargs)
    kbench = types.SimpleNamespace()
    kbench.task = lambda **kwargs: lambda f: MockTask(f)
    kbench.llm = types.SimpleNamespace(prompt=lambda p: '{"final_answer": "0.0"}')
    print('⚠️ kaggle_benchmarks not found. Running in Failsafe (Mock) mode.')

import json
import re
import math
from datetime import datetime

# (Rest of the utils remain the same...)
def extract_json(text):
    if not text: return None
    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence: blob = fence.group(1)
    else:
        start, end = text.find("{"), text.rfind("}")
        if start == -1 or end == -1 or end <= start: return None
        blob = text[start:end + 1]
    try: return json.loads(blob)
    except: return None

def numeric_pass(answer_text, ground_truth, rel_tol=0.015):
    def parse_physics_number(text):
        s = str(text).replace(",", "").strip().lower()
        s = re.sub(r"\\times\s*10\s*(\^|e)\s*{{?(-?\d+)}}?", r"e\2", s)
        s = re.sub(r"\*\s*10\s*(\^|e)\s*{{?(-?\d+)}}?", r"e\2", s)
        match = re.search(r"[-+]?\d*\.?\d+(?:[eE^][-+]?\d+)?", s)
        if match:
            try: return float(match.group(0).replace("^", "e"))
            except: return None
        return None
    pred = parse_physics_number(answer_text)
    try:
        target = float(ground_truth)
        if pred is None: return False
        if target == 0: return abs(pred) < 1e-9
        return math.isclose(pred, target, rel_tol=rel_tol)
    except: return False

# Task 17: Superconducting Ring Max Cooper Speed
TASK_ID = "fp_17"
GROUND_TRUTH = 0.0180

@kbench.task(name="FP-17 Superconducting Ring Max Cooper Speed", description="Physics")
def task_17(llm) -> tuple[int, int]:
    prompt = """You are solving a frontier physics problem. Return valid JSON only.\n\nA superconducting toroidal ring (R=2.5mm, sigma=0.5um^2) rotates at Omega about z. A Josephson junction (Ic=1.8649uA) is present. ns(theta)=n0(1+eps*cos(theta)), eps=0.6, L_geo=200pH, n0=5.58e27 m^-3. Ramp Omega quasistatically from 0. What is the maximum lab-frame tangential speed v_max of the Cooper pairs anywhere on the ring (in m/s) at the instant the n=0 metastable minimum disappears?\n\nReturn JSON: {\"final_answer\": \"<value>\"}"""
    response = llm.prompt(prompt)
    parsed = extract_json(response)
    final_ans = parsed.get("final_answer", "") if parsed else ""
    passed = numeric_pass(final_ans, GROUND_TRUTH)
    return (1 if passed else 0, 1)


In [ ]:
task_17.run(kbench.llm)
